In [0]:
from pyspark.sql import functions as F

catalog = "adb_ecommerce_de_7405607562556202"

customers_df = spark.table(
    f"{catalog}.ecommerce_bronze.customers"
)

In [0]:
customers_df.printSchema()
customers_df.show(5, truncate=False)

# Tranformations

In [0]:
silver_customers = (
    customers_df
    .dropDuplicates(["customer_id"])
    .withColumn("customer_city", F.trim(F.lower(F.col("customer_city"))))
    .withColumn("customer_state", F.upper(F.trim(F.col("customer_state"))))
)

# Write to ADLS Silver

In [0]:
silver_path = (
    "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/silver/customers"
)

silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

#Register it in Unity Catalog

In [0]:
%sql
CREATE TABLE adb_ecommerce_de_7405607562556202.ecommerce_silver.customers
USING DELTA
LOCATION 'abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/silver/customers';

In [0]:
%sql
select * from adb_ecommerce_de_7405607562556202.ecommerce_silver.customers limit 5